# Stage 11 prereq — landmark extraction for the 122-signer paper-split dataset

Walks the 6 paper-split zips (`eng_{train,val,test}_{lex,nonlex}.zip`) and writes a per-clip .npz layout under `/kaggle/working/landmark_cache_122/`.  No training here.

**MediaPipe is pinned** to match the version that produced the 38-signer cache.  A version mismatch silently changes coordinate normalisation; the sanity-check Cell 5 catches that within ±10% feature mean / std.

**Disk discipline** (Kaggle's 20 GB /kaggle/working/ limit + 70 GB /kaggle/temp/):
- Streams from the input zips one at a time via Python's `zipfile` — no full extraction.
- Per-clip output is fp16 .npz (~6 KB each) -> ~250 MB total cache.
- `_DONE` markers per zip survive session disconnects.

**Wall-clock**: ~3 h CPU.  GPU off (MediaPipe is CPU-bound).

## After this kernel commits

Save Version -> Save & Run All.  Then upload `/kaggle/working/landmark_cache_122/` as a new private Kaggle dataset (e.g. `wita-full-english-landmark-cache`).  The Stage 11 training notebook attaches that dataset.

## Cell 1 — Install + clone (MediaPipe pinned)

In [ ]:
%%capture
!pip install editdistance huggingface_hub hgtk scipy --quiet
# PIN MediaPipe to the same version that produced the 38-signer cache.
# If the sanity check (Cell 5) fails the ±10% drift bound, bump this.
!pip install 'mediapipe==0.10.14' --quiet

import sys, os, glob
!rm -rf /kaggle/working/wita_v2
!git clone -b iterative-ablation "https://github.com/Gaurs86/WiTA-v2.git" '/kaggle/working/wita_v2'
sys.path.insert(0, '/kaggle/working')
import mediapipe; print(f'mediapipe version: {mediapipe.__version__}')

## Cell 2 — Locate the 6 paper-split zips

In [ ]:
import os, glob

# ============================================================================
# Hard-coded layout (confirmed by user):
#   /kaggle/input/datasets/<username>/<dataset>/eng_{split}_{subset}/<subset>/<SIGNER>_*/<idx>/<frames>
#
# DATASET_ROOT below points to the parent of the 6 eng_*_* outer dirs.
# `extract_dir_per_clip_landmarks` uses rglob('gt.txt') so the inner
# <subset>/ directory layer is handled automatically.
# ============================================================================

# Detect DATASET_ROOT cheaply — single glob, no deep tree walking.
hits = glob.glob('/kaggle/input/**/eng_train_lex', recursive=True)
assert hits, (
    'Could not find eng_train_lex under /kaggle/input/.\n'
    'Either the dataset is not attached (right panel -> Data -> Add Data),\n'
    'or the dataset uses a different top-level naming.  Set DATASET_ROOT '
    'manually in this cell if needed.'
)
DATASET_ROOT = os.path.dirname(hits[0])
print(f'DATASET_ROOT = {DATASET_ROOT}\n')

# Build the 6 paths.  Each one points at the eng_*_* outer dir; the
# extractor rglobs gt.txt files inside, so any internal layout (with or
# without a <subset>/ middle layer, etc.) is fine.
PATH_TO_LABEL: dict[str, tuple[str, str]] = {}
PATH_TO_KIND:  dict[str, str]              = {}
missing: list[tuple[str, str]] = []
for split in ('train', 'val', 'test'):
    for subset in ('lex', 'nonlex'):
        outer = os.path.join(DATASET_ROOT, f'eng_{split}_{subset}')
        if os.path.isdir(outer):
            PATH_TO_LABEL[outer] = (split, subset)
            PATH_TO_KIND[outer]  = 'dir'
        else:
            missing.append((split, subset))

# Pretty print + count gt.txt files at any depth.
print(f'Classified {len(PATH_TO_LABEL)} source dirs:')
for p in sorted(PATH_TO_LABEL.keys(), key=lambda q: PATH_TO_LABEL[q]):
    s, ss = PATH_TO_LABEL[p]
    n_gt  = len(glob.glob(os.path.join(p, '**', 'gt.txt'), recursive=True))
    print(f'  [{PATH_TO_KIND[p]:<3s}]  {os.path.basename(p):<25s} -> {s}/{ss:<7s}  ({n_gt} gt.txt files)')

if missing:
    print(f'\n!!! Missing dirs for: {missing}\n'
          'Either the dataset is incomplete, or the naming differs.\n'
          'If the names are different, list the actual eng_*_* dirs with:\n'
          '   !ls "{DATASET_ROOT}"\n'
          'then set DATASET_ROOT or rewrite the loop above.')
else:
    print(f'\n✅ Found 6 sources. Cell 3 will set up resume markers.')

# Backward-compat names so Cell 3 / Cell 4 don't change.
classifications = PATH_TO_LABEL
kind_of         = PATH_TO_KIND

## Cell 3 — Output paths + resume markers

In [ ]:
OUT_ROOT = '/kaggle/working/landmark_cache_122'
os.makedirs(OUT_ROOT, exist_ok=True)
MARKER_DIR = os.path.join(OUT_ROOT, '_markers')
os.makedirs(MARKER_DIR, exist_ok=True)

PATH_TO_LABEL = dict(classifications)
PATH_TO_KIND  = dict(kind_of)
assert len(PATH_TO_LABEL) == 6, (
    f'Need 6 sources, got {len(PATH_TO_LABEL)}.  Re-run Cell 2 '
    'with MANUAL_MAPPING set if auto-detect missed.'
)
covered  = sorted(PATH_TO_LABEL.values())
expected = sorted([(s, ss) for s in ('train','val','test') for ss in ('lex','nonlex')])
assert covered == expected, (
    f'Coverage mismatch.\n  expected: {expected}\n  got:      {covered}'
)

def parse_split_subset(p): return PATH_TO_LABEL[p]
def parse_kind(p):         return PATH_TO_KIND[p]
def marker_path(p):
    split, subset = parse_split_subset(p)
    return os.path.join(MARKER_DIR, f'{split}_{subset}_DONE')

print('Final source -> (split, subset) mapping:')
for p, (s, ss) in sorted(PATH_TO_LABEL.items(), key=lambda kv: (kv[1][0], kv[1][1])):
    mk = marker_path(p)
    print(f'  [{parse_kind(p):<3s}]  {os.path.basename(p):<40s} -> {s}/{ss}  '
          f'(marker {"exists" if os.path.exists(mk) else "missing"})')

SPLIT_ORDER = {'val': 0, 'test': 1, 'train': 2}
source_paths = sorted(PATH_TO_LABEL.keys(),
                      key=lambda p: (SPLIT_ORDER[PATH_TO_LABEL[p][0]],
                                     PATH_TO_LABEL[p][1]))

## Cell 4 — Extract per-clip landmarks  (~30 min per zip on Kaggle CPU)

Streams each zip in turn, writing `<SIGNER>__<clip_id>.npz` files.  Resume-aware via `_DONE` markers.

In [ ]:
from wita_v2.datasets.landmark_cache_122 import (
    extract_zip_per_clip_landmarks,
    extract_dir_per_clip_landmarks,
)
from wita_v2.datasets.skeleton_cache  import LandmarkExtractor

extractor = LandmarkExtractor()
all_stats = {}
for p in source_paths:
    split, subset = parse_split_subset(p)
    kind          = parse_kind(p)
    mk            = marker_path(p)
    if os.path.exists(mk):
        print(f'[skip] {split}/{subset} already done')
        continue
    print(f'\n>>> extracting {split}/{subset}  [{kind}]  from {p}')
    if kind == 'zip':
        stats = extract_zip_per_clip_landmarks(
            zip_path=p, out_dir=OUT_ROOT, split=split, subset=subset,
            lang='english', max_frames=64, T_native=32,
            extractor=extractor, overwrite=False,
        )
    else:
        stats = extract_dir_per_clip_landmarks(
            dir_path=p, out_dir=OUT_ROOT, split=split, subset=subset,
            lang='english', max_frames=64, T_native=32,
            extractor=extractor, overwrite=False,
        )
    all_stats[f'{split}_{subset}'] = stats
    with open(mk, 'w') as f:
        import json; json.dump(stats, f, indent=2, default=str)
extractor.close()
print('\nAll sources processed.')

## Cell 5 — Sanity check: feature shape + value range

In [ ]:
import numpy as np
from pathlib import Path
import random

all_npz = list(Path(OUT_ROOT).rglob('*.npz'))
print(f'Total .npz files: {len(all_npz)}')
assert len(all_npz) > 0, 'No clips extracted'

random.seed(42)
sample = random.sample(all_npz, min(100, len(all_npz)))
feats = np.stack([np.load(p, allow_pickle=False)['feature'].astype(np.float32) for p in sample])
print(f'feature shape per clip : {feats.shape[1:]}')
print(f'feature dtype          : {feats.dtype}')
print(f'feature mean           : {feats.mean():.4f}')
print(f'feature std            : {feats.std():.4f}')
print(f'feature min/max        : {feats.min():.4f} / {feats.max():.4f}')
assert feats.shape[1:] == (32, 190), f'Bad shape: {feats.shape[1:]}'
assert np.all(np.isfinite(feats)), 'Non-finite values present'

# Per-split counts.
for split in ('train', 'val', 'test'):
    for subset in ('lex', 'nonlex'):
        n = len(list((Path(OUT_ROOT) / split / subset).glob('*.npz')))
        print(f'  {split}/{subset:<7s}: {n}')

## Cell 6 — Commit kernel + next step

1. **Save Version -> Save & Run All**.  The committed kernel's output dataset contains `landmark_cache_122/`.
2. After it commits, go to **Datasets -> New Dataset -> Notebook Output**, name it `wita-full-english-landmark-cache`.
3. Attach that dataset to the Stage 11 training notebook (next kernel).